# FDIC XGBoost — Local Inference Simulator

**Portfolio demonstration only — not the AWS deployment.** This notebook simulates one local batch-inference run using the  **47-feature XGBoost model**. It does not connect to AWS, trigger scheduled jobs, monitor production drift, or incur AWS charges. It does not retrain the model.

**Example input:** `data/out_of_time.csv` (Q1 2026 snapshots). The model forecasts next-year Q1 EEFFR (Q1 2027), whose actual values may not yet be available. The current-year `EEFFR` is **not** used to predict; it is used only afterward to calculate a potential deterioration warning.

## 1. Set up paths and dependencies

Install packages from the repository-root `requirements.txt` before running. Launch from the repository root or `notebooks/` directory.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "src" / "preprocessing.py").is_file() and (p / "model").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate repository root with src/preprocessing.py and models/. Clone/open the full repository first.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from preprocessing import preprocess_data, FINAL_MODEL_FEATURES

MODEL_PATH = PROJECT_ROOT / "model" / "XGBoost_model.json"
OOT_PATH = PROJECT_ROOT / "data" / "out_of_time.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("XGBoost version:", xgb.__version__)
print("Example input:", OOT_PATH)

Project root: /content
XGBoost version: 3.4.1
Example input: /content/data/out_of_time.csv


## 2. Load the preserved 47-feature model

The saved model is the artifact from the original project, **not** the 45-feature model trained by the refactored comparison notebook. Confirm the exact feature names and order before inference.

In [3]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Missing model: {MODEL_PATH}")
model = xgb.XGBRegressor()
model.load_model(str(MODEL_PATH))
feature_names = model.get_booster().feature_names
if feature_names is None or len(feature_names) != 47:
    raise ValueError("Expected a saved XGBoost model with 47 named predictors.")
if feature_names != FINAL_MODEL_FEATURES:
    raise ValueError("The saved model's feature names/order differ from src/preprocessing.py. Do not score until reconciled.")
if 'EEFFR' in feature_names or 'current_eeffr' in feature_names:
    raise ValueError("Current-year EEFFR must not be an input predictor.")
print("Loaded saved model with", len(feature_names), "features.")
display(pd.DataFrame({"position": range(1, 48), "feature": feature_names}).head(10))

Loaded saved model with 47 features.


,position,feature
0,1,NCLNLSR
1,2,log_assets
2,3,core_deposit_share
3,4,NIMY
4,5,IDNCCIR
5,6,ASTEMPM
6,7,ITAX
7,8,EQCDIV
8,9,EQV
9,10,INTEXPY


## 3. Load out-of-time (OOT) CSV as an example

The CSV must contain the **curated FDIC columns** used by the original training workflow. The preprocessing script creates engineered ratios but does not fetch or reconstruct every upstream FDIC field. This example does not require next-year target values.

In [4]:
if not OOT_PATH.is_file():
    raise FileNotFoundError(f"Missing example CSV: {OOT_PATH}. Place your curated OOT data there or change OOT_PATH.")
raw = pd.read_csv(OOT_PATH, low_memory=False)
if raw.empty:
    raise ValueError("OOT CSV is empty.")
print("OOT shape:", raw.shape)
for col in ('year', 'quarter', 'period'):
    if col in raw.columns:
        print(col, ':', raw[col].dropna().unique()[:10])
display(raw.head(3))

OOT shape: (4307, 80)
year : [2026]
quarter : [1]


,CERT,NAME,SPECGRPDESC,IDT1RWAJR,EXTRA,TFRA,IDNCCONR,EQCDIVNTINC,BKPREM,INTAN,...,IDNCOTHR,RBCRWAJ,IDERNCVR,NFAA,EQTOT,EEFFR,FLAG_NONTRADITIONAL_MODEL,TARGET_EEFFR_NEXT_Q1,year,quarter
0,14,STATE STREET BANK&TRUST CO,ALL OTHER OVER 1 BILLION,14.700159,0,21255711000,0.0,97.763158,3379000,8702000,...,0.000000,15.214463,24.525000,1156752,28711000,71.033520,False,NaN,2026,1
1,35,AUBURNBANK,COMMERCIAL LENDING SPECIALIZATION,16.072707,0,0,0.0,86.586453,44888,754,...,0.000000,17.084645,6.925373,0,90812,67.736702,False,NaN,2026,1
2,39,ROBERTSON BANKING CO,COMMERCIAL LENDING SPECIALIZATION,14.437776,0,0,0.0,15.644904,6366,0,...,0.144939,15.690086,0.000000,0,49314,48.623161,False,NaN,2026,1


## 4. Apply the original inference preprocessing

`src/preprocessing.py` generates the financial ratios and log-assets, removes excluded fields, and returns the exact **47 predictors in saved-model order**. Missing numerical cells are left missing for XGBoost native handling. Missing required columns cause an explicit error.

In [5]:
X, metadata, target = preprocess_data(raw)
assert list(X.columns) == feature_names
assert X.shape == (len(raw), 47)
print("Inference feature matrix:", X.shape)
print("Missing feature cells (%):", round(100 * X.isna().mean().mean(), 2))
print("Optional next-year target present:", not target.empty)
display(X.head(3))

Inference feature matrix: (4307, 47)
Missing feature cells (%): 0.01
Optional next-year target present: True


,NCLNLSR,log_assets,core_deposit_share,NIMY,IDNCCIR,ASTEMPM,ITAX,EQCDIV,EQV,INTEXPY,...,NTLNLSR,INTINCY,securities_to_assets,NTLNLS,FLAG_NONTRADITIONAL_MODEL,DEPDOM,NPERFV,ELNATR,OBSDIR,NCRER
0,0.437530,19.772761,0.673114,1.240520,0.0,7.817538,207000,743000,7.427577,2.440618,...,0.332188,3.681138,0.279240,40000,0,217874000,0.055880,16000,3892123000,9.342561
1,0.053089,13.841528,0.913758,3.346074,0.0,7.331136,616,1943,8.847976,1.135954,...,0.280245,4.482029,0.220960,402,0,932742,0.030106,-76,21929,0.062718
2,0.067681,13.071497,0.924501,4.216904,0.0,6.600042,212,393,10.377460,1.110094,...,-0.041171,5.326999,0.215295,-32,0,413964,0.044192,0,0,0.069006


## 5. Predict next-year Q1 efficiency ratio

The original model uses a pseudo-Huber regression objective and predicts EEFFR on its original scale. **Do not apply `expm1`** to these predictions.

In [6]:
config = model.get_booster().save_config()
if 'reg:pseudohubererror' not in config:
    raise ValueError("Saved model objective differs from the expected pseudo-Huber objective. Verify target scale before inference.")
predictions = np.asarray(model.predict(X), dtype=float)
if not np.isfinite(predictions).all():
    raise ValueError("Predictions contain non-finite values.")
results = metadata.reset_index(drop=True).copy()
results['predicted_next_year_eeffr'] = predictions
if 'year' in results.columns:
    years = pd.to_numeric(results['year'], errors='coerce')
    results['prediction_for_year'] = years + 1
print("Scored", len(results), "banks/records")
display(results.head(10))

Scored 4307 banks/records


,CERT,NAME,year,quarter,SPECGRPDESC,predicted_next_year_eeffr,prediction_for_year
0,14,STATE STREET BANK&TRUST CO,2026,1,ALL OTHER OVER 1 BILLION,73.758835,2027
1,35,AUBURNBANK,2026,1,COMMERCIAL LENDING SPECIALIZATION,67.351913,2027
2,39,ROBERTSON BANKING CO,2026,1,COMMERCIAL LENDING SPECIALIZATION,57.450653,2027
3,41,PHENIX-GIRARD BANK,2026,1,OTHER SPECIALIZED UNDER 1 BILLION,65.054688,2027
4,49,BANK OF EVERGREEN,2026,1,ALL OTHER UNDER 1 BILLION,65.614929,2027
5,50,COMMUNITY SPIRIT BANK,2026,1,COMMERCIAL LENDING SPECIALIZATION,65.163765,2027
6,51,BANK OF VERNON,2026,1,COMMERCIAL LENDING SPECIALIZATION,46.002827,2027
7,52,CITIZENS BANK OF WINFIELD,2026,1,OTHER SPECIALIZED UNDER 1 BILLION,68.895439,2027
8,54,EXCHANGE BANK OF ALABAMA,2026,1,COMMERCIAL LENDING SPECIALIZATION,51.856651,2027
9,58,UNITED BANK,2026,1,COMMERCIAL LENDING SPECIALIZATION,60.346783,2027


## 6. Simulated early-warning rule

A bank is **flagged for review** when predicted next-year EEFFR exceeds its current-year EEFFR by **more than 10 percentage points**. Current-year EEFFR is read from the original CSV **only after** prediction. If unavailable, the notebook exports forecasts without flags. These flags are screening signals, not confirmed deterioration.

In [7]:
if 'EEFFR' in raw.columns:
    current = pd.to_numeric(raw['EEFFR'], errors='coerce').reset_index(drop=True)
    results['current_eeffr'] = current
    results['predicted_change_pp'] = results['predicted_next_year_eeffr'] - current
    results['early_warning_gt_10pp'] = (results['predicted_change_pp'] > 10).astype('boolean')
    results.loc[results['predicted_change_pp'].isna(), 'early_warning_gt_10pp'] = pd.NA
    print('Records with available current EEFFR:', int(current.notna().sum()))
    print('Flagged for review:', int(results['early_warning_gt_10pp'].fillna(False).sum()))
    display(results.sort_values('predicted_change_pp', ascending=False).head(15))
else:
    print('EEFFR not found: no deterioration flags can be calculated; predictions are still valid.')
    display(results.head(10))

Records with available current EEFFR: 4307
Flagged for review: 375


,CERT,NAME,year,quarter,SPECGRPDESC,predicted_next_year_eeffr,prediction_for_year,current_eeffr,predicted_change_pp,early_warning_gt_10pp
3552,32574,MONET BANK,2026,1,ALL OTHER OVER 1 BILLION,155.967300,2027,28.571043,127.396257,True
4161,58687,TOUCHMARK NATIONAL BANK,2026,1,COMMERCIAL LENDING SPECIALIZATION,155.089005,2027,85.071199,70.017806,True
16,90,COMMERCIAL BANK&TRUST CO,2026,1,COMMERCIAL LENDING SPECIALIZATION,137.993835,2027,68.440367,69.553468,True
4142,58590,NANO BANC,2026,1,COMMERCIAL LENDING SPECIALIZATION,176.477890,2027,111.583012,64.894878,True
2970,25796,COMMUNITY B&T WEST GEORGIA,2026,1,COMMERCIAL LENDING SPECIALIZATION,154.767075,2027,92.382271,62.384803,True
2259,16583,STATE BANK OF BURRTON,2026,1,ALL OTHER UNDER 1 BILLION,149.143555,2027,87.387387,61.756167,True
1846,13986,TRANSACT BANK NATIONAL ASSN,2026,1,COMMERCIAL LENDING SPECIALIZATION,153.041229,2027,97.373358,55.667871,True
3638,34069,WILMINGTON TRUST NA,2026,1,OTHER SPECIALIZED UNDER 1 BILLION,151.699585,2027,99.584718,52.114867,True
595,4173,PAULS VALLEY NATIONAL BANK,2026,1,COMMERCIAL LENDING SPECIALIZATION,118.977409,2027,67.941606,51.035804,True
3023,26539,MICRONESIA,2026,1,OTHER SPECIALIZED UNDER 1 BILLION,116.638443,2027,66.122041,50.516402,True


## 7. Export simulator results

This writes a local CSV; it does not upload data to S3 or invoke any AWS service.

In [8]:
output_path = OUTPUT_DIR / 'xgboost_oot_inference_simulation.csv'
results.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Rows:', len(results))

Saved: /content/outputs/xgboost_oot_inference_simulation.csv
Rows: 4307


## Interpretation and reproducibility notes

- **Purpose:** An optional, no-AWS-cost simulator for GitHub readers who want to try the preserved model. **It is not a reproduction or live demonstration of the paid AWS environment.**
- **Model:** `models/xgboost_model.json` (47 features); no retraining. The separate training notebook may select 45 features and is not interchangeable with this saved artifact.
- **Preprocessing:** `src/preprocessing.py` contains the 47-feature ordered schema and engineering logic. Upstream curated FDIC field definitions, units, and specialisation flag must match training.
- **OOT example:** Q1 2026 input produces Q1 2027 forecasts. Without actual next-year labels, MAE/RMSE/R² **cannot** be computed on OOT data.
- **Warning flags:** `predicted_next_year_eeffr - current_eeffr > 10`; current-year EEFFR is used after prediction only.
- **Not simulated:** AWS Batch, S3 storage, annual scheduling, PSI drift checks, SNS alerts, and production monitoring.
- **Dependencies:** The supplied `requirements.txt` specifies `xgboost==3.4.1`; preserve environment versions for reproducibility.